# 03 — Research Questions Starter

Example analysis patterns for menstrual health research questions, demonstrated on synthetic data.

**Important caveat:** these patterns show *how* to structure an analysis, not *what is true*. The synthetic generator builds in a few artificial relationships (e.g. heavier flow tends to have higher pain scores) purely so charts are not flat. No finding here is real.

When these patterns are eventually applied to approved real data, every output must pass the privacy review in `governance/` — including small-cell suppression for any published aggregate.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("..") / "synthetic-data" / "datasets" / "survey_responses_1000.csv"
if not DATA_PATH.exists():
    DATA_PATH = Path("synthetic-data") / "datasets" / "survey_responses_1000.csv"

df = pd.read_csv(DATA_PATH)
len(df)

## Pattern 1 — Is symptom burden associated with missed school or work?

A crosstab of pain score bands against attendance impact. With real data this would need confidence intervals and confounder discussion; here it demonstrates the shape of the analysis.

In [ ]:
df["pain_band"] = pd.cut(df["pain_score"], bins=[-1, 3, 6, 10], labels=["low (0-3)", "medium (4-6)", "high (7-10)"])
pd.crosstab(df["pain_band"], df["missed_school_or_work"], normalize="index").round(3)

## Pattern 2 — Does product access differ by setting?

Useful for infrastructure and supply questions. Note how the controlled vocabulary makes the comparison clean.

In [ ]:
pd.crosstab(df["setting"], df["product_access"], normalize="index").round(3)

## Pattern 3 — Care-seeking gaps

`wanted_but_could_not_access` is the key access-gap category. Aggregating by country shows how a gap metric could be reported — with real data, small groups would need suppression before publication.

In [ ]:
gap = (
    df.assign(care_gap=df["healthcare_access"] == "wanted_but_could_not_access")
    .groupby("country_code")
    .agg(records=("record_id", "count"), care_gap_rate=("care_gap", "mean"))
    .round(3)
)
gap

## Pattern 4 — Small-cell suppression

Before publishing any aggregate from real data, suppress cells below a minimum count so individuals in small groups cannot be singled out. Here is the minimal version of that pattern:

In [ ]:
MIN_CELL = 20

cell_counts = df.groupby(["country_code", "age_band"]).size().rename("count").reset_index()
published = cell_counts.copy()
published.loc[published["count"] < MIN_CELL, "count"] = None  # suppressed
published.pivot(index="country_code", columns="age_band", values="count")

## Build on this

- Turn one of these patterns into a reusable, documented function
- Add uncertainty reporting (bootstrap intervals) to a pattern
- Build a research question library (see `issues/challenge_statements.md`)
- Write the plain-language summary that would accompany a real version of one of these analyses